In [ ]:
import pandas as pd
from rdkit import Chem
import networkx as nx

In [ ]:
# Function to convert SMILES to molecular graphs
def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)  # Convert SMILES to RDKit molecule object
    G = nx.Graph()  # Initialize NetworkX graph
    
    # Add nodes (atoms)
    for atom in mol.GetAtoms():
        G.add_node(atom.GetIdx(), element=atom.GetSymbol())
    
    # Add edges (bonds)
    for bond in mol.GetBonds():
        G.add_edge(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx(), bond_type=bond.GetBondType())
    
    return G

In [ ]:
# Function to create a new column with graphs
def add_graph_column(df, smiles_column):
    # Apply the smiles_to_graph function to each SMILES string in the column
    df['molecule_graph'] = df[smiles_column].apply(smiles_to_graph)
    return df

In [ ]:
data = pd.read_csv('../../data/processed/Combined_df.csv')

In [ ]:
data.shape

In [ ]:
data.columns

In [ ]:
# Add the molecule_graph column
df_with_graphs = add_graph_column(data, 'SMILES')

In [ ]:
df_with_graphs.columns

In [ ]:
import torch
from torch_geometric.data import Data
import networkx as nx
import numpy as np

In [ ]:
from rdkit import Chem

# Assume `df` is your pandas DataFrame with a 'smiles' column
def get_unique_atom_types(smiles_list):
    atom_types = set()
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        for atom in mol.GetAtoms():
            atom_types.add(atom.GetSymbol())
    return sorted(list(atom_types))

# Get unique atom types from the SMILES column
unique_atom_types = get_unique_atom_types(df_with_graphs['SMILES'])
print(unique_atom_types)

In [ ]:
# Convert NetworkX graph to PyTorch Geometric Data object
def networkx_to_pyg_data(G):
    # Extract node features (e.g., atom types)
    node_features = []
    for node in G.nodes(data=True):
        # One-hot encode the atom type
        atom_type = node[1]['element']
        feature_vector = [1 if atom_type == e else 0 for e in ['Al', 'B', 'Br', 'C', 'Cl', 'F', 'Ge', 'I', 'N', 'O', 'P', 'S', 'Se', 'Si', 'Te', 'Zn']]
        node_features.append(feature_vector)

    x = torch.tensor(node_features, dtype=torch.float)

    # Extract edges and edge features (if any)
    edge_index = []
    for edge in G.edges(data=True):
        edge_index.append([edge[0], edge[1]])
        edge_index.append([edge[1], edge[0]])  # Since the graph is undirected

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    # Create a PyTorch Geometric Data object
    data = Data(x=x, edge_index=edge_index)

    return data

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.nn import global_mean_pool

# Define a GNN Model
class GNN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GNN, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)
        # Define fully connected (FC) layers for final output
        self.fc1 = torch.nn.Linear(hidden_dim, hidden_dim)
        self.fc2 = torch.nn.Linear(hidden_dim, output_dim)
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        # Apply graph convolutions with ReLU activations
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = global_mean_pool(x, batch)
        # Apply fully connected layers
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Example usage:
input_dim = 16  # One-hot encoding for atom types (C, O, N, F, Cl, Br)
hidden_dim = 16
output_dim = 8 # Output dimension (e.g., number of classes or features to predict)

model = GNN(input_dim, hidden_dim, output_dim)

In [ ]:
# Step 2: Convert NetworkX graphs to PyTorch Geometric Data objects
df_with_graphs['pyg_data'] = df_with_graphs['molecule_graph'].apply(networkx_to_pyg_data)

In [ ]:
data = df_with_graphs['pyg_data'][0]  # Get the first molecule's graph

In [ ]:
data

In [ ]:
output = model(data)

In [ ]:
print(output)

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.data import DataLoader

In [ ]:
# Define the MSE loss function
def mse_loss(output, target):
    return F.mse_loss(output, target)

In [ ]:
# Training loop
def train(model, data_loader, optimizer):
    model.train()  # Set the model to training mode
    total_loss = 0
    for data in data_loader:
        optimizer.zero_grad()  # Clear gradients from the previous step
        
        output = model(data)  # Forward pass: Get predictions from the model
        
        # Calculate the loss (Mean Squared Error)
        loss = mse_loss(output.squeeze(), data.y)
        total_loss += loss.item()  # Accumulate the loss
        
        loss.backward()  # Backpropagation: Compute the gradients
        optimizer.step()  # Update the model parameters using the optimizer
        
    return total_loss / len(data_loader)

In [ ]:
# Evaluation (optional) to monitor R² score during training
from sklearn.metrics import r2_score

def r2_metric(output, target):
    output = output.detach().cpu().numpy()
    target = target.detach().cpu().numpy()
    return r2_score(target, output)

def evaluate(model, data_loader):
    model.eval()  # Set the model to evaluation mode
    r2_scores = []
    with torch.no_grad():  # Disable gradient calculation for evaluation
        for data in data_loader:
            output = model(data)
            r2 = r2_metric(output, data.y)  # Compute R² score for evaluation
            r2_scores.append(r2)
    return sum(r2_scores) / len(r2_scores)

In [ ]:
# Assuming 'model', 'data_loader', and 'optimizer' are already defined:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
df_with_graphs.columns

In [ ]:
import torch
from torch_geometric.data import Data

# Assume `df` is your pandas DataFrame with SMILES and target columns

def create_data_object(row):
    """
    Converts a row of the pandas DataFrame (containing molecule and target info)
    into a PyTorch Geometric Data object.
    """
    graph_data = row['pyg_data']  # This is the PyTorch Geometric data (from the SMILES graph)
    target = row['PCE']  # The target value (e.g., PCE efficiency) to predict
    
    # Add the target to the Data object
    graph_data.y = torch.tensor([target], dtype=torch.float)
    
    return graph_data

# Convert the entire DataFrame into a list of Data objects
data_list = df_with_graphs.apply(create_data_object, axis=1).tolist()

In [ ]:
from torch_geometric.data import DataLoader

# Define the DataLoader
data_loader = DataLoader(data_list, batch_size=32, shuffle=True)

In [ ]:
# Training loop for multiple epochs
num_epochs = 10
for epoch in range(num_epochs):
    avg_loss = train(model, data_loader, optimizer)  # Training step
    r2_score_value = evaluate(model, data_loader)  # Optional evaluation for R²
    print(f"Epoch {epoch+1}, Loss: {avg_loss}, R² Score: {r2_score_value}")